# 🧹 02 · Visual Data Prep com Linguagem Natural (Lakeflow Designer + Genie Code)

**Objetivo:** transformar uma exportação "suja" de vendas das lojas em uma tabela confiável, **sem escrever código** — só conversando com o Genie Code e conferindo cada passo no canvas visual.

⏱️ **Duração:** 30–40 min  |  ✅ **Pré-requisito:** ter rodado o notebook `01_dados_sinteticos`

---
### 🎯 O problema de negócio
> *"A diretoria comercial quer saber **quais lojas bateram a meta de receita em cada mês**. O problema: o relatório de vendas vem de um sistema antigo, cheio de erros de digitação, datas em formatos diferentes e vendas duplicadas. Hoje alguém passa dois dias por mês arrumando isso no Excel."*

### 📦 Tabelas que vamos usar (no seu catálogo/schema)
| Tabela | Papel |
|---|---|
| `vendas_lojas_bruto` | Vendas brutas do sistema legado (**a bagunça**) |
| `lojas` | Cadastro das lojas (nome, cidade, região, tipo) |
| `metas_lojas` | Meta mensal de cada loja |

💡 **Dica:** rode as células cinzas (botão ▶️ ou `Shift + Enter`) só para **enxergar** os dados. Você não precisa entender o código — ele já está pronto.

### ⚙️ Configuração — rode esta célula primeiro (ou clique em **Executar tudo**)
Use o **mesmo Catálogo e Schema** do notebook 01 (campos no topo do notebook; padrão no Free Edition: `workspace` / `workshop_negocio`).
As caixas roxas 📋 ao longo do notebook mostram os prompts **já com o nome das suas tabelas** — é só copiar e colar.

In [0]:
import html
dbutils.widgets.text("catalogo", "workspace", "Catálogo")
dbutils.widgets.text("schema", "workshop_negocio", "Schema")
dbutils.widgets.text("seu_nome", "SEU_NOME", "Seu nome (sufixo da tabela)")
CATALOGO, SCHEMA = dbutils.widgets.get("catalogo").strip(), dbutils.widgets.get("schema").strip()
NOME = dbutils.widgets.get("seu_nome").strip().lower().replace(" ", "_")
T = f"{CATALOGO}.{SCHEMA}"
spark.sql(f"USE CATALOG `{CATALOGO}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

def prompt(texto, titulo="📋 Copie e cole"):
    """Mostra um prompt/SQL pronto para copiar, trocando {T} por catálogo.schema e {NOME} pelo seu nome."""
    corpo = html.escape(texto.strip().replace("{T}", T).replace("{NOME}", NOME))
    displayHTML(f"""<div style="font-family:sans-serif;font-size:12px;font-weight:bold;color:#6929C4;margin:2px 0">{titulo}</div>
<pre style="background:#F4F0FB;border-left:4px solid #6929C4;padding:10px 14px;white-space:pre-wrap;font-size:13px;margin:0">{corpo}</pre>""")

prompt(f"Catálogo: {CATALOGO}\nSchema:   {SCHEMA}\nSuas tabelas: {T}.<tabela>\nSua tabela final: {T}.vendas_tratadas_{NOME}", "✅ Configuração ativa")

## 👀 Passo 0 · Conheça a bagunça
Olhe as colunas `data_venda`, `cod_loja`, `produto`, `valor_venda` e `status_venda`. Quantos problemas você consegue encontrar?

In [0]:
%sql
SELECT * FROM vendas_lojas_bruto LIMIT 20

In [0]:
%sql
-- Quantas formas diferentes de escrever o mesmo produto e o mesmo status?
SELECT 'produto' AS coluna, produto AS valor, count(*) AS qtd FROM vendas_lojas_bruto GROUP BY ALL
UNION ALL
SELECT 'status_venda', coalesce(status_venda, '(vazio)'), count(*) FROM vendas_lojas_bruto GROUP BY ALL
ORDER BY coluna, qtd DESC

<details><summary>🕵️ <b>Clique para ver a lista de problemas</b></summary>

| # | Coluna | Problema | Exemplo |
|---|---|---|---|
| 1 | `data_venda` | 3 formatos diferentes, um deles com hora | `2025-03-14`, `14/03/2025`, `14-03-2025 10:22` |
| 2 | `cod_loja` | Espaços e letras minúsculas | `" LJ-0012"`, `lj-0012` |
| 3 | `produto` | 13 grafias para 5 categorias | `Pós-Pago`, `pos pago`, `POS PAGO` |
| 4 | `valor_venda` | Texto, com "R$", vírgula ou ponto decimal, e vazios | `R$ 1.234,56`, `129.90`, `129,90` |
| 5 | `uf` | Maiúsculas e minúsculas misturadas | `sp`, `SP` |
| 6 | `status_venda` | Grafias diferentes e vazios | `Aprovada`, `APROVADA`, `aprovada` |
| 7 | (linha toda) | ~3% das vendas estão **duplicadas** | mesmo `id_venda` duas vezes |
</details>

---
## 🚀 Passo 1 · Crie o seu Visual Data Prep

1. No menu lateral clique em **+ Novo** → **Visual data prep** (Lakeflow Designer).
2. Dê um nome: **`Vendas Lojas - <seu nome>`**.
3. Abra o painel do **Genie Code** (ícone ✨ no canto da tela) — é por ele que vamos "conversar".
4. Em cada passo abaixo: **copie o prompt**, cole no Genie Code, e **confira o resultado** clicando no novo bloco do canvas (a pré-visualização mostra as primeiras linhas).

> 🧠 **Regra de ouro:** um pedido por vez. Pedidos pequenos e claros = blocos fáceis de conferir. Se algo sair errado, peça para corrigir ("*No último passo, ...*") ou desfaça.

---
## 🗣️ Passo 2 · Roteiro de prompts
### Prompt 1 — Trazer os dados

In [0]:
prompt("Adicione como fonte a tabela {T}.vendas_lojas_bruto.", "📋 Prompt 1")

✅ **Confira:** aparece um bloco de fonte com ~29.900 linhas e 9 colunas.

---
### Prompt 2 — Remover duplicatas
```
Remova as linhas duplicadas. Considere duplicada qualquer venda que tenha o mesmo id_venda, mantendo apenas uma.
```
✅ **Confira:** sobram **29.000** linhas.

---
### Prompt 3 — Arrumar as datas
```
A coluna data_venda está em formatos diferentes: "2025-03-14", "14/03/2025" e "14-03-2025 10:22".
Crie uma coluna data_venda do tipo data que entenda os três formatos e ignore a hora.
```
✅ **Confira:** nenhuma data ficou vazia e todas estão entre jan/2025 e set/2026.

---
### Prompt 4 — Padronizar o código da loja e a UF
```
Padronize as colunas cod_loja e uf: remova espaços no começo e no fim e deixe tudo em letras maiúsculas.
Renomeie cod_loja para loja_id.
```

---
### Prompt 5 — Converter o valor de texto para número
```
A coluna valor_venda é texto e vem em formatos como "R$ 1.234,56", "129,90" e "129.90".
Converta para número decimal em reais. No formato com "R$", o ponto é separador de milhar e a vírgula é decimal.
```
✅ **Confira:** valores entre ~15 e ~470. Se aparecerem valores como 12.990 ou 1,29, peça a correção ao Genie Code explicando o erro.

---
### Prompt 6 — Agrupar os produtos em categorias
```
Crie uma coluna categoria a partir de produto, agrupando as diferentes grafias nestas 5 categorias:
Pós-pago, Controle, Pré-pago, Fibra e Combo. Ignore maiúsculas/minúsculas, acentos e espaços.
```
✅ **Confira:** só restam 5 valores distintos. (Peça: *"mostre a contagem por categoria"*.)

---
### Prompt 7 — Padronizar o status e filtrar só as vendas válidas
```
Padronize status_venda (primeira letra maiúscula, sem espaços) e mantenha apenas as vendas com status "Aprovada".
```

---
### Prompt 8 — Agregar por loja e mês
```
Agrupe por loja_id e pelo mês da data_venda (primeiro dia do mês, chame de mes).
Calcule qtd_vendas (quantidade de vendas) e receita_vendas (soma do valor_venda, arredondada em 2 casas).
```

---
### Prompt 9 — Enriquecer com o cadastro das lojas

In [0]:
prompt("""Junte com a tabela {T}.lojas pela coluna loja_id
para trazer nome_loja, tipo_canal, cidade, uf e regiao.""", "📋 Prompt 9")

### Prompt 10 — Comparar com a meta

In [0]:
prompt("""Junte com a tabela {T}.metas_lojas usando loja_id e mes (left join).
Crie pct_atingimento_meta = receita_vendas / meta_receita
e bateu_meta = verdadeiro quando receita_vendas >= meta_receita.""", "📋 Prompt 10")

### Prompt 11 — Salvar o resultado
Preencha o campo **"Seu nome"** no topo do notebook e rode de novo a célula de configuração para o nome aparecer abaixo.

In [0]:
prompt("Salve o resultado final como a tabela {T}.vendas_tratadas_{NOME}.", "📋 Prompt 11")

Depois clique em **Executar** (Run) para materializar a tabela.

---
## ✅ Passo 3 · Confira seu resultado contra o gabarito
As duas linhas devem ficar **iguais** (ou muito próximas).

In [0]:
minha = f"{T}.vendas_tratadas_{NOME}"
consulta = """SELECT '{rotulo}' AS origem, count(*) AS linhas_loja_mes, sum(qtd_vendas) AS vendas_aprovadas,
                     round(sum(receita_vendas), 2) AS receita_total, count_if(bateu_meta) AS lojas_mes_que_bateram_meta
              FROM {tabela}"""
partes = [consulta.format(rotulo="🎯 Gabarito", tabela=f"{T}.vendas_lojas_tratadas")]
if spark.catalog.tableExists(minha):
    partes.append(consulta.format(rotulo="🙋 Meu resultado", tabela=minha))
else:
    print(f"⚠️ A tabela {minha} ainda não existe. Termine o Prompt 11 e execute o Data Prep.")
display(spark.sql(" UNION ALL ".join(partes)))

---
## 🏆 Desafios extras (para quem terminar antes)
```
Adicione uma coluna ticket_medio = receita_vendas / qtd_vendas.
```
```
Crie uma coluna faixa_atingimento: "Abaixo de 80%", "80% a 100%" e "Acima de 100%".
```
```
Quantas vendas foram descartadas em cada passo? Mostre um resumo.
```
```
Explique em português, passo a passo, o que este fluxo faz — como se fosse para um gerente comercial.
```

### 💬 Para discutir
- Quanto tempo esse tratamento levaria no Excel? E todo mês?
- O fluxo pode ser **agendado** (Lakeflow Jobs) para rodar sozinho todo dia, e é governado pelo Unity Catalog (quem pode ver, linhagem, histórico).
- Por trás de cada bloco existe código gerado — o time de dados consegue revisar e versionar no Git.

➡️ **Próximo:** `03_dashboards_com_genie_code` — vamos transformar esses dados em um dashboard executivo.